In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
DATA_PATH = Path("../data/raw/cars.csv")

print("Dataset exists:", DATA_PATH.exists())
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

Dataset exists: True
File size: 138.49 MB


In [3]:
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 762091
Columns: 20


In [4]:
df.head()

,manufacturer,model,year,mileage,engine,transmission,drivetrain,fuel_type,mpg,exterior_color,interior_color,accidents_or_damage,one_owner,personal_use_only,seller_name,seller_rating,driver_rating,driver_reviews_num,price_drop,price
0,Acura,ILX Hybrid 1.5L,2013,92945.0,"1.5L I-4 i-VTEC variable valve control, engine...",Automatic,Front-wheel Drive,Gasoline,39-38,Black,Parchment,0.0,0.0,0.0,Iconic Coach,NaN,4.4,12.0,300.0,13988.0
1,Acura,ILX Hybrid 1.5L,2013,47645.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Gray,Ebony,1.0,1.0,1.0,Kars Today,NaN,4.4,12.0,NaN,17995.0
2,Acura,ILX Hybrid 1.5L,2013,53422.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Bellanova White Pearl,Ebony,0.0,1.0,1.0,Weiss Toyota of South County,4.3,4.4,12.0,500.0,17000.0
3,Acura,ILX Hybrid 1.5L,2013,117598.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,Polished Metal Metallic,NaN,0.0,1.0,1.0,Apple Tree Acura,NaN,4.4,12.0,675.0,14958.0
4,Acura,ILX Hybrid 1.5L,2013,114865.0,1.5L I4 8V MPFI SOHC Hybrid,Automatic CVT,Front-wheel Drive,Hybrid,39-38,NaN,Ebony,1.0,0.0,1.0,Herb Connolly Chevrolet,3.7,4.4,12.0,300.0,14498.0


In [5]:
df.columns.tolist()

['manufacturer',
 'model',
 'year',
 'mileage',
 'engine',
 'transmission',
 'drivetrain',
 'fuel_type',
 'mpg',
 'exterior_color',
 'interior_color',
 'accidents_or_damage',
 'one_owner',
 'personal_use_only',
 'seller_name',
 'seller_rating',
 'driver_rating',
 'driver_reviews_num',
 'price_drop',
 'price']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 762091 entries, 0 to 762090
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   manufacturer         762091 non-null  str    
 1   model                762091 non-null  str    
 2   year                 762091 non-null  int64  
 3   mileage              761585 non-null  float64
 4   engine               747041 non-null  str    
 5   transmission         752187 non-null  str    
 6   drivetrain           740529 non-null  str    
 7   fuel_type            739164 non-null  str    
 8   mpg                  620020 non-null  str    
 9   exterior_color       753232 non-null  str    
 10  interior_color       705116 non-null  str    
 11  accidents_or_damage  737879 non-null  float64
 12  one_owner            730608 non-null  float64
 13  personal_use_only    737239 non-null  float64
 14  seller_name          753498 non-null  str    
 15  seller_rating        548118 

In [7]:
missing_pct = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

missing_pct

price_drop             46.185954
seller_rating          28.077093
mpg                    18.642262
interior_color          7.476141
driver_rating           4.150685
one_owner               4.131134
personal_use_only       3.261028
accidents_or_damage     3.177048
fuel_type               3.008433
drivetrain              2.829321
engine                  1.974830
transmission            1.299582
exterior_color          1.162460
seller_name             1.127556
mileage                 0.066396
driver_reviews_num      0.000000
manufacturer            0.000000
model                   0.000000
year                    0.000000
price                   0.000000
dtype: float64

## Manufacturer and Model Coverage

The next step is to determine whether the dataset contains enough listings for the enthusiast and performance vehicles targeted by Apex Analytics.

In [8]:
manufacturer_counts = df["manufacturer"].value_counts()

manufacturer_counts.head(50)

manufacturer
Ford             79526
Toyota           59535
Chevrolet        56043
Nissan           48529
Jeep             41665
Mercedes-Benz    40824
Honda            37612
BMW              37570
Kia              35063
GMC              29563
Dodge            25250
Subaru           24767
Volkswagen       24620
Hyundai          22203
Lexus            21301
RAM              19364
Audi             17863
Cadillac         17794
Mazda            15485
Buick            14624
Chrysler         12647
INFINITI         12289
Land Rover       12272
Porsche          11461
Lincoln          10608
Volvo            10029
Acura             8489
Tesla             5883
Mitsubishi        5743
Jaguar            3469
Name: count, dtype: int64

In [9]:
candidate_brands = [
    "BMW",
    "Mercedes-Benz",
    "Porsche",
    "Audi",
    "Chevrolet",
    "Toyota",
    "Nissan",
    "Cadillac"
]

df[df["manufacturer"].isin(candidate_brands)]["manufacturer"].value_counts()

manufacturer
Toyota           59535
Chevrolet        56043
Nissan           48529
Mercedes-Benz    40824
BMW              37570
Audi             17863
Cadillac         17794
Porsche          11461
Name: count, dtype: int64

## BMW Performance Model Coverage

Inspect BMW model naming conventions and determine the number of usable listings for M2, M3, M4, and M5 models.

In [10]:
bmw_models = (
    df.loc[df["manufacturer"] == "BMW", "model"]
      .value_counts()
)

bmw_models.head(100)

model
X3 xDrive30i                             1481
330 i xDrive                             1476
X5 xDrive40i                             1391
330 i                                    1271
X5 xDrive35i                              972
                                         ... 
428 Gran Coupe i xDrive                    85
228 Gran Coupe 228i sDrive Gran Coupe      82
530e 530e xDrive                           82
650 Gran Coupe i xDrive                    82
i3 94 Ah w/Range Extender                  81
Name: count, Length: 100, dtype: int64

In [11]:
bmw_performance = df[
    (df["manufacturer"] == "BMW") &
    (
        df["model"]
        .str.contains(r"\b(M2|M3|M4|M5)\b", case=False, na=False, regex=True)
    )
].copy()

bmw_performance["apex_model"] = (
    bmw_performance["model"]
    .str.extract(r"\b(M2|M3|M4|M5)\b", expand=False)
    .str.upper()
)

bmw_performance["apex_model"].value_counts()

/var/folders/bz/54m5dhc13mz38tytyd8_bmbm0000gn/T/ipykernel_2865/1462650962.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(r"\b(M2|M3|M4|M5)\b", case=False, na=False, regex=True)


apex_model
M4    565
M3    528
M5    396
M2    157
Name: count, dtype: int64

In [12]:
bmw_year_counts = (
    bmw_performance
    .groupby(["apex_model", "year"])
    .size()
    .unstack(fill_value=0)
)

bmw_year_counts

year,1988,1990,1991,1993,1995,1996,1997,1998,1999,2000,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
apex_model,,,,,,,,,,,,,,,,,,,,,
M2,0,0,0,0,0,0,0,0,0,0,...,0,0,7,33,38,9,47,23,0,0
M3,2,2,1,0,3,2,8,7,12,0,...,0,28,30,31,57,0,0,43,60,36
M4,0,0,0,0,0,0,0,0,0,0,...,0,66,79,28,51,39,86,43,131,42
M5,3,0,2,2,0,0,0,0,0,2,...,13,16,12,0,34,96,72,38,26,13
